# Reproducing Gu et al. (2011) from CRCNS stc-1

**Paper.** Gu Y, Liu S, Fetsch CR, Yang Y, Fok S, Sunkara A, DeAngelis GC, Angelaki DE (2011).
Perceptual learning reduces interneuronal correlations in macaque visual cortex. *Neuron* 71:750-761.
[doi:10.1016/j.neuron.2011.06.015](https://doi.org/10.1016/j.neuron.2011.06.015). The claims file
was written from the version of record (journal PDF text, with the Experimental Procedures and the
legends of Figures 1 to 8) and the Supplemental Information (legends of Figures S1 to S8).

**Data set.** CRCNS [stc-1](https://doi.org/10.6080/K07P8WKF) (doi:10.6080/K07P8WKF), data
descriptor version 0.6 (5 September 2018), read from the public bucket `crcnsarchive` under
`stc-1/live/`. CRCNS data sets are not versioned, and the paper cites no version, so the version is
recorded as `unversioned` (docs/archives.md, CRCNS). The bucket holds one copy of the data,
`stc-1/live/` (`orig/` holds only an older description document), so `live/` is the version
pinned. The file read, `MSTd.mat`, is checked on loading against its MD5 sum in the data set's
`checksums.md5`, `cd4c13b360aa05689015f1bd2b88a05a`, so a later change makes the notebook fail
(`VIP.mat`, read only to check for overlap, `8d7924e0a1ae86c56b78cd1e2f530e2d`).

**What the data set holds.** Derived values only. For the paper, experiment 3 of `MSTd.mat` holds
the 127 MSTd pairs separated by less than 1 mm, with per condition (vestibular, visual) the signal
correlation, the noise correlation and the two units' preferred headings, and the recording's
`file_id`, which names the monkey. It holds no spike times or trial responses, no training-group
label, no distances, no tuning curves of the pair units, none of the 52 pairs over 1 mm, none of the
single-electrode database of Figure 4, Figure S4 and Figure 8A, no training histories and no eye
records. Experiments 1 and 2 hold the tuning curves, choice probabilities and neuronal thresholds
of 129 single units of two trained monkeys from Gu et al. (2008); they are used for
fig4dh-lateral-more-sensitive and fig8ab-choice-probability-lateral as a subset of the paper's
database.

**Outside inputs.** None beyond the paper itself: the training group of each monkey comes from the
paper's per-animal pair counts (Figure S2B legend), checked by the link claim
`training-group-link`.

**Random numbers.** Only the bootstrap intervals of fig5-intercept-lower-trained and
fig5-regression-coefficients draw random numbers, each from its own generator with a fixed seed.

| Claim | Tested here |
|---|---|
| training-group-link | tested (link) |
| fig2b-trained-lower-noise-correlation | tested |
| figs2b-individual-animals | tested (means per animal; no time courses) |
| fig1-example-pair | tested |
| fig1f-modality-correlated | tested on the 127 pairs under 1 mm |
| fig1f-modality-means-equal | tested on the 127 pairs under 1 mm |
| fig5-noise-signal-positive | tested |
| fig5-slope-unchanged | tested |
| fig5-intercept-lower-trained | tested |
| fig5-regression-coefficients | tested |
| fig6a-population-thresholds | tested (simulation) |
| fig6b-slope-not-intercept | tested (simulation) |
| fig7a-pair-sign-of-effect | tested (simulation) |
| fig7c-narrow-preferences | tested (simulation) |
| fig4dh-lateral-more-sensitive | tested on 129 units of two trained monkeys, without the training factor |
| fig8ab-choice-probability-lateral | tested on 129 units of two monkeys |
| fig8cd-noise-not-preference | tested |
| fig2c-timecourse-naive-higher, figs2a-difference-decreases, fig3-mean-response-unchanged, fig3-fano-unchanged, figs8c-renormalization, discussion-stimulus-reduces-correlation | blocked: no spike times or trial responses |
| figs1-distance | blocked: no distances, no pairs over 1 mm |
| methods-single-vs-multiple-electrodes | blocked: no electrode label |
| fig2a-behavioral-learning | blocked: no training histories |
| fig4-tuning-width-visual, fig4-tuning-width-vestibular, fig4-amplitude-unchanged, fig4dh-training-no-sensitivity-change, figs4-population-sensitivity-any-reference | blocked: no naive animals in the single-unit data |
| fig7b-msdt-pairs, figs5-congruency | blocked: no tuning curves of the pair units |
| figs6-task-independent | blocked: no discrimination-task correlations |
| figs8ab-eye-movements | blocked: no eye records |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

import stc1

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
started = time.time()
mat = stc1.load_mat("MSTd.mat")
vip = stc1.load_mat("VIP.mat")
print(f"CRCNS {stc1.DATASET_ID} ({stc1.DATASET_VERSION}, doi:{stc1.DATASET_DOI}): MSTd.mat and VIP.mat "
      f"read and checked against their MD5 sums in {time.time() - started:.1f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
published = stc1.checksums()
print("checksums.md5 now lists:", published)
print("MSTd.mat and VIP.mat match the sums pinned in stc1.py:",
      all(published[k] == v for k, v in stc1.MD5.items()))
print("Variables in MSTd.mat:", [k for k in mat if not k.startswith("__")], mat["__header__"].decode())
for e in ("experiment1", "experiment2", "experiment3"):
    x = mat[e]
    print(f"{e}: name {x.name!r}, fields {x._fieldnames}")
q = mat["experiment3"].pairs[0]
print("experiment3.pairs fields:", q._fieldnames, "; per condition:", q.ves._fieldnames)
u1, u2 = mat["experiment1"].units[0], mat["experiment2"].units[0]
print("experiment1.units fields:", u1._fieldnames, u1.ves._fieldnames)
print("experiment2.units fields:", u2._fieldnames, u2.ves._fieldnames, "; com:", u2.com._fieldnames)
print("experiment2.behv.subj:", [(s.monk_id, s.ves.thresh._fieldnames) for s in mat["experiment2"].behv.subj])

In [ ]:
pairs = stc1.pairs_table(mat)
print("Pairs in experiment3:", len(pairs), "(data descriptor: 127 MSTd pairs; paper: 127 pairs under 1 mm)")
print("Missing values:", int(pairs.isna().sum().sum()))
cols = ["rsig_ves", "rsig_vis", "rnoise_ves", "rnoise_vis", "pref1_ves", "pref2_ves", "pref1_vis", "pref2_vis"]
print("\nRanges (correlations are unitless; preferred headings in degrees):")
print(pairs[cols].describe().loc[["count", "mean", "min", "max"]].round(3).to_string())
print("\nPairs per monkey (monkey from file_id):")
counts = pairs.monkey.value_counts()
print(counts.to_string())
print("Monkeys in the data:", sorted(counts.index.tolist()),
      "; data descriptor lists MSTd pair monkeys 1, 2, 3, 5, 17, 18, 19")
print("Paper: 8 monkeys in all (Experimental Procedures, Subjects)")

The pairs hold every field the descriptor names, with no missing values. Correlations lie within
-1 and 1 and preferred headings within -180 and 180 degrees. The `file_id`s name eight monkeys,
as many as the paper's eight; the data descriptor lists seven for the MSTd pairs and leaves out
monkey 24, which has two pairs.

In [ ]:
# Units per recording: a unit in several pairs of one recording has the same preferred headings.
per_rec = pairs.groupby("file_id").size()
units = stc1.neurons_per_recording(pairs)
shape = pd.DataFrame({"pairs": per_rec, "units": units})
print("Recordings by (pairs, units):", shape.value_counts().sort_index().to_dict())
print("Every recording with k units has k(k-1)/2 pairs:",
      bool((shape.pairs == shape.units * (shape.units - 1) // 2).all()))
print("Distinct units over the 127 pairs:", int(units.sum()))
print("Largest number of pairs in one recording:", int(per_rec.max()), list(per_rec[per_rec == per_rec.max()].index),
      "; paper: 'maximum of 5 pairs in an experiment' (Experimental Procedures)")
dup = pairs.duplicated(subset=["rsig_ves", "rsig_vis", "rnoise_ves", "rnoise_vis"], keep=False)
print("\nPairs with identical correlations (deposited twice):", int(dup.sum()))
vip_ids = {p.file_id for p in np.atleast_1d(vip["experiment3"].pairs)}
vip_vals = {(round(float(p.ves.corr_noise), 4), round(float(p.vis.corr_noise), 4)) for p in np.atleast_1d(vip["experiment3"].pairs)}
mst_vals = set(zip(pairs.rnoise_ves.round(4), pairs.rnoise_vis.round(4)))
print("VIP pairs:", len(np.atleast_1d(vip["experiment3"].pairs)), "; recordings shared with MSTd:", len(vip_ids & set(pairs.file_id)),
      "; noise-correlation pairs shared:", len(vip_vals & mst_vals))

Within each recording the units can be told apart by their preferred headings, and every recording
with k units has exactly k(k-1)/2 pairs, so the pairs are all pairs of the units recorded together
and the 127 pairs come from the distinct units counted above. No pair is deposited twice, and no
MSTd pair is repeated in `VIP.mat`. One recording holds six pairs (four units), more than the
paper's stated maximum of five pairs per experiment.

In [ ]:
# Signal correlation should follow the similarity of the two preferences: a check that the
# preferred headings and the signal correlation of a pair refer to the same two units and condition.
for cond in ("ves", "vis"):
    sim = np.cos(np.deg2rad(pairs[f"pref1_{cond}"] - pairs[f"pref2_{cond}"]))
    other = "vis" if cond == "ves" else "ves"
    sim_other = np.cos(np.deg2rad(pairs[f"pref1_{other}"] - pairs[f"pref2_{other}"]))
    print(f"{cond}: Spearman rho of rsignal with cos(preference difference) of the same condition "
          f"{stats.spearmanr(pairs[f'rsig_{cond}'], sim)[0]:.3f}, of the other condition {stats.spearmanr(pairs[f'rsig_{cond}'], sim_other)[0]:.3f}")

Signal correlation follows the difference between the two preferred headings of the same
condition much more closely than that of the other condition, so the fields are consistent with
their description. The data hold no time series, so there is no clock to check against a trial
table (protocol, Author, step 2): every value used here is a per-pair or per-unit summary.

In [ ]:
groups = stc1.assign_groups(counts)
pairs["group"] = pairs.monkey.map(groups)
pairs["rnoise_pooled"] = stc1.pooled_rnoise(pairs)
units_by_group = {g: int(units[pairs[pairs.group == g].file_id.unique()].sum()) for g in ("trained", "naive")}
print("Group assigned per monkey:", groups)
print("Pairs per group:", pairs.group.value_counts().to_dict(), "; units per group:", units_by_group)
print("\nSingle-unit experiments: monkeys", sorted({stc1.monkey_of(u.file_id) for u in mat["experiment1"].units}),
      "; units in experiment1 and experiment2:", len(mat["experiment1"].units), len(mat["experiment2"].units))
same_cells = all(a.file_id.split("r")[0] == b.file_id.split("r")[0]
                 for a, b in zip(mat["experiment1"].units, mat["experiment2"].units))
print("experiment1 and experiment2 list the same cells in the same order (cell id before 'r'):", same_cells)
print("Headings of experiment1 tuning curves (deg):", mat["experiment1"].units[0].ves.stim_global)
print("Headings of experiment2 (deg):", mat["experiment2"].units[0].ves.stim_local)
print("Behavioural thresholds held (experiment2.behv, vestibular, visual):",
      {s.monk_id: (float(s.ves.thresh.mu), float(s.vis.thresh.mu)) for s in mat["experiment2"].behv.subj})

The data set gives no training group. The link claim below assigns the groups from the paper's
per-animal pair counts. Experiments 1 and 2 list the same 129 cells, from monkeys 2 and 5, which
the data set's own behavioural thresholds show performed the heading discrimination task.
These thresholds, one per monkey and condition, are the only behavioural values in the data set.
For fig2a-behavioral-learning, which is blocked for want of the training histories, they are the
only part of the claim that can be compared: both vestibular values (monkey 2 1.2 degrees,
monkey 5 3.1 degrees, printed above) lie inside the criterion's plateau range of 0.9 to 3.3
degrees, as do the visual ones (1.2 and 3.25 degrees). Whether they are plateau values, and the
fall from above 10 degrees, cannot be checked.

## training-group-link

> The monkeys of the data set's 127 MSTd pairs can be assigned to the paper's naive and trained
> groups, and the assignment agrees with what the data set and the paper both describe.

Criterion (added by the author): match monkeys to the Figure S2B counts (trained 35, 26, 18, 8 and
2; naive 22, 11 and 5); check 89 and 38 pairs, 146 and 64 neurons (Figure 3 legend), and that the
monkeys with behavioural thresholds in the data set are trained.

In [ ]:
legend = sorted(n for ns in stc1.PAPER_PAIRS_PER_ANIMAL.values() for n in ns)
print("Pairs per monkey in the data:", sorted(counts.values.tolist()))
print("Pairs per animal in the Figure S2B legend:", legend)
print("Counts in the legend all distinct:", len(set(legend)) == len(legend))
print("Monkeys matched one to one:", len(groups) == len(counts), groups)
print("Pairs per group:", pairs.group.value_counts().to_dict(), "(paper: trained 89, naive 38)")
print("Units per group:", units_by_group, "(Figure 3 legend: trained 146, naive 64)")
behv = sorted(int(s.monk_id) for s in mat["experiment2"].behv.subj)
print("Monkeys with discrimination thresholds in experiment2:", behv, "-> groups", [groups[m] for m in behv])

**Result: reproduced.** The eight monkeys' pair counts (35, 26, 22, 18, 11, 8, 5, 2) equal the eight
counts of the Figure S2B legend, all distinct, so each monkey matches one count. The match gives
89 trained and 38 naive pairs, and 146 trained and 64 naive units, as the paper and the Figure 3
legend state. Monkeys 2 and 5, whose discrimination thresholds the data set holds, fall in the
trained group. The groups rest on the paper's counts. The data set itself does not label them.

## fig2b-trained-lower-noise-correlation

> Mean noise correlation among MSTd pairs (separated by less than 1 mm) is significantly smaller in
> trained animals (0.023 +- 0.017 SEM, n = 89) than in naive animals (0.116 +- 0.031, n = 38).

The pooled rnoise is the mean of the vestibular and visual values (assumption, see `stc1.py`). The
criterion's readings are a two-sided Student and a Welch t test; whether the stored values include
the renormalization in blocks of 20 trials cannot be told from the data set, which holds one value
per condition and no trial responses, so that reading cannot be run. The per-condition values are
shown as variants.

In [ ]:
def group_ttest(col, df=pairs):
    t, n = df[df.group == "trained"][col], df[df.group == "naive"][col]
    return {"measure": col, "trained_mean": t.mean(), "trained_sem": t.sem(), "n_trained": len(t),
            "naive_mean": n.mean(), "naive_sem": n.sem(), "n_naive": len(n),
            "difference": n.mean() - t.mean(),
            "p_student": stats.ttest_ind(t, n).pvalue, "p_welch": stats.ttest_ind(t, n, equal_var=False).pvalue}

fig2b = pd.DataFrame([group_ttest(c) for c in ("rnoise_pooled", "rnoise_ves", "rnoise_vis")])
print(fig2b.round(4).to_string(index=False))
print("\nExpected: trained 0.023 +- 0.017 (0.006 to 0.040), naive 0.116 +- 0.031 (0.085 to 0.147), p = 0.006")
row = fig2b.iloc[0]
print("Pooled means within tolerance:", abs(row.trained_mean - 0.023) <= 0.017, abs(row.naive_mean - 0.116) <= 0.031)
print("\nWithout the trained animal with two pairs (monkey 24):")
print(pd.DataFrame([group_ttest("rnoise_pooled", pairs[pairs.monkey != 24])]).round(4).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(8, 3), sharex=True)
for a, g in zip(ax, ("naive", "trained")):
    v = pairs[pairs.group == g].rnoise_pooled
    a.hist(v, bins=np.arange(-0.6, 0.71, 0.05), color="tab:blue" if g == "naive" else "tab:red")
    a.axvline(v.mean(), color="k")
    a.set_title(f"{g}, n = {len(v)}")
    a.set_xlabel("pooled rnoise")
plt.tight_layout(); plt.show()
print("Range of pooled rnoise plotted:", pairs.rnoise_pooled.min().round(3), "to", pairs.rnoise_pooled.max().round(3), "(all inside the axes)")

**Result: reproduced, resting on the assumption that the pooled rnoise is the mean of the two
per-condition values.** Trained 0.0241 +- 0.0174 (n = 89) and naive 0.1190 +- 0.0316 (n = 38), both
within tolerance of 0.023 +- 0.017 and 0.116 +- 0.031, Student t test p = 0.0056 (paper 0.006),
Welch p = 0.0108. The visual values alone give the same result (p = 0.0047). The vestibular values
alone do not reach significance (p = 0.071). Leaving out the two-pair animal gives p = 0.0032.

## figs2b-individual-animals

> The greater noise correlation in naive animals holds for individual animals; four trained animals
> (35, 26, 18 and 8 pairs) and three naive animals (22, 11 and 5 pairs) are shown, and the
> difference is consistent across animals within each group.

Per animal, the mean pooled rnoise of its pairs. The time courses of Figure S2B need spike times,
which the data set does not hold.

In [ ]:
per_animal = (pairs.groupby(["group", "monkey"]).rnoise_pooled.agg(["count", "mean", "sem"])
              .sort_values(["group", "count"], ascending=[True, False]).round(4))
print(per_animal.to_string())
shown = per_animal[per_animal["count"] > 2]
naive_means = shown.loc["naive", "mean"]
trained_means = shown.loc["trained", "mean"]
print("\nPair counts, trained shown:", shown.loc["trained", "count"].tolist(), "(paper 35, 26, 18, 8)",
      "; naive:", shown.loc["naive", "count"].tolist(), "(paper 22, 11, 5)")
print("Lowest naive animal mean:", naive_means.min(), "; highest trained animal mean (shown):", trained_means.max())
print("Every naive mean above every trained mean (seven animals shown):", naive_means.min() > trained_means.max())
print("Every naive mean above the trained group mean", round(pairs[pairs.group == "trained"].rnoise_pooled.mean(), 4), ":",
      bool((naive_means > pairs[pairs.group == "trained"].rnoise_pooled.mean()).all()))
print("\nPer condition (variants):")
print(pairs.groupby(["group", "monkey"])[["rnoise_ves", "rnoise_vis"]].mean().round(4).to_string())

**Result: reproduced for the per-animal means; the time courses cannot be tested.** The pair counts
of the animals shown are 35, 26, 18 and 8 (trained) and 22, 11 and 5 (naive), as in the legend. Every
naive animal's mean (0.0876 to 0.2545) is above every trained animal's mean among the four shown
(-0.0449 to 0.0484). The trained animal with two pairs, not shown in the paper, has a mean of 0.2708.

## fig1-example-pair

> For the example pair of Figure 1, signal correlation is 0.83 (visual) and 0.79 (vestibular), and
> noise correlation is 0.29 (visual, p = 0.04) and 0.14 (vestibular, p = 0.3).

The paper does not identify the pair; the criterion asks whether any pair has all four values
within 10%. The p values need the number of trials, which the data set does not hold.

In [ ]:
target = {"rsig_vis": 0.83, "rsig_ves": 0.79, "rnoise_vis": 0.29, "rnoise_ves": 0.14}
inside = pd.DataFrame({k: (pairs[k] - v).abs() <= 0.1 * v + 1e-9 for k, v in target.items()})
pairs["n_within"] = inside.sum(axis=1)
print("Pairs by number of the four values within tolerance:", pairs.n_within.value_counts().sort_index().to_dict())
cand = pairs[pairs.n_within >= 2].copy()
dist = sum(((cand[k] - v) / v) ** 2 for k, v in target.items()) ** 0.5
print("\nPairs with at least two values within tolerance, closest first:")
print(cand.assign(rel_distance=dist).sort_values("rel_distance")[["file_id", "group", *target, "n_within", "rel_distance"]].round(3).head(10).to_string(index=False))
print("\nPairs with all four within tolerance:", int((pairs.n_within == 4).sum()))

**Result: reproduced.** One pair, m2c338r1 (a trained animal), has all four values within
tolerance: rsignal 0.828 visual and 0.786 vestibular, rnoise 0.291 visual and 0.142 vestibular. No
other pair has more than two. The per-pair p values (0.04 and 0.3) need trial counts, which the
data set does not hold.

## fig1f-modality-correlated

> Noise correlations computed from visual and vestibular responses are correlated across pairs
> (R = 0.38, p << 0.001, Spearman rank correlation, 179 pairs).

Only the 127 pairs under 1 mm are in the data set, the criterion's other reading.

In [ ]:
rho, p = stats.spearmanr(pairs.rnoise_vis, pairs.rnoise_ves)
print(f"Spearman R = {rho:.3f}, p = {p:.2e}, n = {len(pairs)} (expected 0.38 +- 0.038, i.e. 0.342 to 0.418, n = 179)")
print("Pearson r =", round(stats.pearsonr(pairs.rnoise_vis, pairs.rnoise_ves)[0], 3))
print("\nPer group and per monkey:")
for key, g in list(pairs.groupby("group")) + list(pairs.groupby("monkey")):
    if len(g) >= 5:
        r_, p_ = stats.spearmanr(g.rnoise_vis, g.rnoise_ves)
        print(f"  {key}: n = {len(g)}, R = {r_:.3f}, p = {p_:.3g}")
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.scatter(pairs.rnoise_vis, pairs.rnoise_ves, s=10, c=np.where(pairs.group == "naive", "tab:blue", "tab:red"))
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_xlabel("rnoise visual"); ax.set_ylabel("rnoise vestibular")
plt.show()

**Result: partially reproduced.** Over the 127 pairs under 1 mm (the 179 are not in the data set),
Spearman R = 0.449, p = 1.2e-07. The correlation is positive and significant, but R is outside
0.342 to 0.418.

## fig1f-modality-means-equal

> Mean noise correlation does not differ between the vestibular (0.035 +- 0.014 SEM) and visual
> (0.039 +- 0.015) conditions (p > 0.8, paired t test).

A claim of absence, judged on a paired t test at p < 0.05, on the 127 pairs under 1 mm.

In [ ]:
v, w = pairs.rnoise_ves, pairs.rnoise_vis
res = stats.ttest_rel(v, w)
print(f"vestibular {v.mean():.4f} +- {v.sem():.4f}, visual {w.mean():.4f} +- {w.sem():.4f} (n = {len(v)})")
print(f"difference (vestibular - visual) {(v - w).mean():.4f}, 95% CI {np.round(stats.t.interval(0.95, len(v) - 1, (v - w).mean(), (v - w).sem()), 4)}")
print(f"paired t = {res.statistic:.3f}, p = {res.pvalue:.3f}")
print("Expected: vestibular 0.035 +- 0.014 (0.021 to 0.049), visual 0.039 +- 0.015 (0.024 to 0.054)")
print("Within tolerance:", abs(v.mean() - 0.035) <= 0.014, abs(w.mean() - 0.039) <= 0.015)
for g, d in pairs.groupby("group"):
    print(f"  {g}: vestibular {d.rnoise_ves.mean():.4f}, visual {d.rnoise_vis.mean():.4f}, paired p = {stats.ttest_rel(d.rnoise_ves, d.rnoise_vis).pvalue:.3f}")

**Result: reproduced.** The paired t test finds no difference (t = 0.346, p = 0.730, n = 127).
Vestibular 0.0559 +- 0.0173 is outside the tolerance of 0.035 +- 0.014, and visual 0.0491 +- 0.0199
is inside 0.039 +- 0.015. The sample is the 127 pairs under 1 mm, not the paper's 179.

## fig5-noise-signal-positive

> Noise correlation increases with signal correlation in both conditions (vestibular p = 0.0001,
> visual p = 0.0004, ANCOVA).

Per condition, an ANCOVA of rnoise on rsignal and training group with their interaction, over the
127 pairs; the rnoise readings are the pooled value and the value of the same condition. How the
sums of squares are formed is not stated; Type II tests of the model with the interaction, and of
the model without it (the classical ANCOVA), are both shown (a choice made here, since the paper
does not say). The tests of all three ANCOVA claims are computed in this cell.

In [ ]:
def ancova(cond, measure):
    s = f"rsig_{cond}"
    full = smf.ols(f"{measure} ~ {s} * C(group)", pairs).fit()
    add = smf.ols(f"{measure} ~ {s} + C(group)", pairs).fit()
    a_full, a_add = sm.stats.anova_lm(full, typ=2), sm.stats.anova_lm(add, typ=2)
    return {"condition": cond, "rnoise": measure,
            "slope_all": add.params[s],
            "p_rsignal_full": a_full.loc[s, "PR(>F)"], "p_rsignal_additive": a_add.loc[s, "PR(>F)"],
            "p_group_full": a_full.loc["C(group)", "PR(>F)"], "p_group_additive": a_add.loc["C(group)", "PR(>F)"],
            "p_interaction": a_full.loc[f"{s}:C(group)", "PR(>F)"],
            "group_effect_additive": add.params["C(group)[T.trained]"]}

anc = pd.DataFrame([ancova(c, m) for c in ("ves", "vis") for m in ("rnoise_pooled", f"rnoise_{c}")])
print(anc.round(5).to_string(index=False))
print("\nPaper: rsignal vestibular p = 0.0001, visual p = 0.0004; interaction 0.9 and 0.9; training 0.02 and 0.008")
print("\nSlope of pooled rnoise on rsignal per monkey (OLS; monkeys with at least 5 pairs):")
rows = []
for m, d in pairs.groupby("monkey"):
    if len(d) >= 5:
        for c in ("ves", "vis"):
            r = stats.linregress(d[f"rsig_{c}"], d.rnoise_pooled)
            rows.append({"monkey": m, "group": groups[m], "condition": c, "n": len(d), "slope": r.slope, "p": r.pvalue})
print(pd.DataFrame(rows).pivot_table(index=["group", "monkey", "n"], columns="condition", values=["slope", "p"]).round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
for a, c in zip(ax, ("vis", "ves")):
    for g, col in (("naive", "tab:blue"), ("trained", "tab:red")):
        d = pairs[pairs.group == g]
        a.scatter(d[f"rsig_{c}"], d.rnoise_pooled, s=10, color=col, label=g)
        fit = stats.linregress(d[f"rsig_{c}"], d.rnoise_pooled)
        a.plot([-1, 1], fit.intercept + fit.slope * np.array([-1, 1]), color=col)
    a.set_xlim(-1, 1); a.set_ylim(-1, 1); a.set_xlabel(f"rsignal {c}")
ax[0].set_ylabel("pooled rnoise"); ax[0].legend(); plt.tight_layout(); plt.show()

**Result: reproduced, resting on the pooled-rnoise assumption.** With the pooled rnoise the rsignal
term is significant in both conditions: vestibular p = 0.00017, visual p = 0.00070 (paper 0.0001 and
0.0004), with positive slopes (0.130 and 0.105). With the same-condition rnoise it is also
significant (p = 0.00009 and 0.017). Per monkey, the slopes are positive in 12 of 14 monkey and
condition cells. The two negative cells are both monkey 3, which has 5 pairs.

## fig5-slope-unchanged

> Training does not change the slope of the relation between noise and signal correlation
> (vestibular p = 0.9, visual p = 0.9, ANCOVA interaction).

A claim of absence, judged on the interaction term at p < 0.05 with the pooled rnoise; slopes and
their 95% intervals per group, from separate fits and from the ANCOVA model with the interaction
(Figure 5 legend: "Lines represent regression fits (ANCOVA)"), are reported.

In [ ]:
def group_lines(cond, measure):
    s = f"rsig_{cond}"
    rows = []
    cell = smf.ols(f"{measure} ~ 0 + C(group) + C(group):{s}", pairs).fit()
    ci_cell = cell.conf_int()
    for g in ("naive", "trained"):
        sep = smf.ols(f"{measure} ~ {s}", pairs[pairs.group == g]).fit()
        ci = sep.conf_int()
        rows.append({"condition": cond, "rnoise": measure, "group": g,
                     "slope": sep.params[s], "slope_lo": ci.loc[s, 0], "slope_hi": ci.loc[s, 1],
                     "intercept": sep.params["Intercept"], "int_lo": ci.loc["Intercept", 0], "int_hi": ci.loc["Intercept", 1],
                     "int_lo_ancova": ci_cell.loc[f"C(group)[{g}]", 0], "int_hi_ancova": ci_cell.loc[f"C(group)[{g}]", 1],
                     "slope_lo_ancova": ci_cell.loc[f"C(group)[{g}]:{s}", 0], "slope_hi_ancova": ci_cell.loc[f"C(group)[{g}]:{s}", 1]})
    return rows

lines = pd.DataFrame([r for c in ("ves", "vis") for m in ("rnoise_pooled", f"rnoise_{c}") for r in group_lines(c, m)])
print(lines.round(4).to_string(index=False))
print("\nInteraction p (pooled rnoise):", anc[anc.rnoise == "rnoise_pooled"].set_index("condition").p_interaction.round(3).to_dict())
print("Interaction p (same-condition rnoise):", anc[anc.rnoise != "rnoise_pooled"].set_index("condition").p_interaction.round(3).to_dict())

**Result: reproduced, resting on the pooled-rnoise assumption.** The interaction of rsignal and
training is not significant: vestibular p = 0.830, visual p = 0.962 (paper 0.9 and 0.9). Per group the
slopes are naive 0.119 [-0.021, 0.259] and trained 0.135 [0.060, 0.210] (vestibular), naive 0.107
[-0.026, 0.240] and trained 0.104 [0.038, 0.170] (visual). With the same-condition rnoise the
interaction p values are 0.630 and 0.894.

## fig5-intercept-lower-trained

> Training lowers noise correlation uniformly across signal correlations, a main effect of training
> (vestibular p = 0.02, visual p = 0.008, ANCOVA) with non-overlapping 95% confidence intervals of
> the intercepts.

The training main effect is in the table of fig5-noise-signal-positive; the intercept intervals
are from the fig5-slope-unchanged table, under both the separate fits and the ANCOVA model.

In [ ]:
for (c, m), d in lines.groupby(["condition", "rnoise"], sort=False):
    n, t = d.set_index("group").loc["naive"], d.set_index("group").loc["trained"]
    a = anc[(anc.condition == c) & (anc.rnoise == m)].iloc[0]
    print(f"{c}, {m}: training p (full / additive) {a.p_group_full:.4f} / {a.p_group_additive:.4f}; "
          f"intercept naive {n.intercept:.4f}, trained {t.intercept:.4f}")
    print(f"    separate fits: naive [{n.int_lo:.4f}, {n.int_hi:.4f}], trained [{t.int_lo:.4f}, {t.int_hi:.4f}], overlap {t.int_hi >= n.int_lo}")
    print(f"    ANCOVA model:  naive [{n.int_lo_ancova:.4f}, {n.int_hi_ancova:.4f}], trained [{t.int_lo_ancova:.4f}, {t.int_hi_ancova:.4f}], overlap {t.int_hi_ancova >= n.int_lo_ancova}")
print("\nVariant, intercepts of the additive model (common slope), 95% CI of each group's intercept:")
for c in ("ves", "vis"):
    add = smf.ols(f"rnoise_pooled ~ 0 + C(group) + rsig_{c}", pairs).fit()
    ci = add.conf_int()
    print(f"  {c}: naive [{ci.loc['C(group)[naive]', 0]:.4f}, {ci.loc['C(group)[naive]', 1]:.4f}], "
          f"trained [{ci.loc['C(group)[trained]', 0]:.4f}, {ci.loc['C(group)[trained]', 1]:.4f}]")
print("\nVariant, bootstrap 95% intervals of each group's intercept from its own fit (pooled rnoise,"
      " 2000 resamples of pairs within the group):")
rng_int = np.random.default_rng(2011)
for c in ("ves", "vis"):
    ci = {}
    for g, d in pairs.groupby("group"):
        X = np.column_stack([np.ones(len(d)), d[f"rsig_{c}"].values])
        y = d.rnoise_pooled.values
        b = [np.linalg.lstsq(X[k], y[k], rcond=None)[0][0]
             for k in (rng_int.integers(0, len(d), len(d)) for _ in range(2000))]
        ci[g] = np.percentile(b, [2.5, 97.5])
    print(f"  {c}: naive [{ci['naive'][0]:.4f}, {ci['naive'][1]:.4f}], trained [{ci['trained'][0]:.4f}, "
          f"{ci['trained'][1]:.4f}], overlap {ci['trained'][1] >= ci['naive'][0]}")


**Result: partially reproduced, resting on the pooled-rnoise assumption.** The main effect of
training holds in both conditions with the pooled rnoise (vestibular p = 0.0184, visual p = 0.0074;
paper 0.02 and 0.008). The naive intercepts are higher (0.095 against 0.015, and 0.096 against
0.009). But the 95% intervals of the intercepts overlap under every reading. With separate fits,
vestibular naive is [0.026, 0.163] and trained [-0.018, 0.048]. In the ANCOVA model they are
[0.036, 0.153] and [-0.020, 0.050]. The visual intervals overlap the same way, as do those of the
additive model, and the bootstrap intervals of each group's own fit (vestibular naive [0.040,
0.151], trained [-0.016, 0.046]; visual [0.021, 0.172] and [-0.023, 0.040]). With the same-condition rnoise the vestibular main effect is not significant
(p = 0.183).

## fig5-regression-coefficients

> The plane fitted to noise and the two signal correlations is rnoise = 0.12 rsignal,vestibular +
> 0.091 rsignal,visual + 0.072 in naive animals, with the same slopes and an intercept of 0.005 in
> trained animals.

Type II (orthogonal) plane fits of pooled rnoise on the two signal correlations, under the
criterion's readings (a), (b) and (c); ordinary least squares as a variant. 95% intervals of the
intercepts by a bootstrap over pairs within each group (2000 resamples, seed fixed; the number of
resamples is a standard choice).

In [ ]:
X_cols = ["rsig_ves", "rsig_vis"]
expected = np.array([0.12, 0.091, 0.072, 0.005])
tol = np.array([0.012, 0.0091, 0.0072, 0.0005])

def readings(df):
    out = {}
    nv, tr = df[df.group == "naive"], df[df.group == "trained"]
    cen = df.copy()
    for c in X_cols + ["rnoise_pooled"]:
        cen[c] = df[c] - df.groupby("group")[c].transform("mean")
    coef_a, _ = stc1.tls_plane(cen[X_cols].values, cen.rnoise_pooled.values)
    out["a: common slopes, centred per group"] = coef_a
    coef_b, _ = stc1.tls_plane(df[X_cols].values, df.rnoise_pooled.values)
    out["b: slopes of one fit to all pairs"] = coef_b
    coef_c, _ = stc1.tls_plane(nv[X_cols].values, nv.rnoise_pooled.values)
    out["c: slopes of the naive fit"] = coef_c
    res = {}
    for k, coef in out.items():
        b_n = nv.rnoise_pooled.mean() - nv[X_cols].mean().values @ coef
        b_t = tr.rnoise_pooled.mean() - tr[X_cols].mean().values @ coef
        res[k] = np.array([coef[0], coef[1], b_n, b_t])
    for g, d in (("naive", nv), ("trained", tr)):
        co, b = stc1.tls_plane(d[X_cols].values, d.rnoise_pooled.values)
        res[f"separate orthogonal fit, {g}"] = np.array([co[0], co[1], b if g == "naive" else np.nan, b if g == "trained" else np.nan])
    ols_all = smf.ols("rnoise_pooled ~ 0 + C(group) + rsig_ves + rsig_vis", df).fit().params
    res["variant: OLS, common slopes"] = np.array([ols_all["rsig_ves"], ols_all["rsig_vis"], ols_all["C(group)[naive]"], ols_all["C(group)[trained]"]])
    for g, d in (("naive", nv), ("trained", tr)):
        o = smf.ols("rnoise_pooled ~ rsig_ves + rsig_vis", d).fit().params
        res[f"variant: OLS, {g}"] = np.array([o["rsig_ves"], o["rsig_vis"], o["Intercept"] if g == "naive" else np.nan, o["Intercept"] if g == "trained" else np.nan])
    return res

fits = readings(pairs)
table = pd.DataFrame(fits, index=["a_vestibular", "a_visual", "b_naive", "b_trained"]).T
within = table.apply(lambda r: pd.Series(np.abs(r.values - expected) <= tol + 1e-12, index=[f"in_{i}" for i in table.columns]), axis=1)
print(table.round(4).join(within).to_string())
print("\nExpected:", expected, "+-", tol)

rng = np.random.default_rng(20110825)
boot = {k: [] for k in fits if k[0] in "abc"}
for _ in range(2000):
    sample = pd.concat([d.iloc[rng.integers(0, len(d), len(d))] for _, d in pairs.groupby("group")])
    r = readings(sample)
    for k in boot:
        boot[k].append(r[k])
print("\nBootstrap 95% intervals (2000 resamples within groups):")
for k, v in boot.items():
    lo, hi = np.percentile(np.array(v), [2.5, 97.5], axis=0)
    print(f"  {k}: " + ", ".join(f"{n} [{a:.4f}, {b:.4f}]" for n, a, b in zip(table.columns, lo, hi)))

**Result: partially reproduced, resting on the pooled-rnoise assumption.** Under reading (a) the
slopes are 0.119 and 0.084, and the naive intercept is 0.077, all within tolerance. The trained
intercept is 0.0034, outside 0.005 +- 0.0005. Its bootstrap interval is [-0.026, 0.035], which
includes 0.005. Reading (b) gives 0.132, 0.085, 0.074 and 0.0024, with the same pattern. Under
reading (c) the visual slope (0.106) is outside tolerance. No reading puts all four values within
tolerance. The slopes match and the naive intercept exceeds the trained one, which the criterion
calls partial. The separate orthogonal fits come close to the paper's two equations
(naive 0.123, 0.106, 0.071; trained intercept 0.0046). The criterion does not list them as a
reading, and they do not share slopes.

## fig6a-population-thresholds

> In simulated populations, correlated noise as measured in naive animals raises the population
> threshold about 5-fold over independent noise at 2000 neurons, while the uniform reduction in
> trained animals has little effect on the threshold compared with naive.

Equations 2 to 5 with the paper's coefficients, wrapped-Gaussian tuning of 135 degrees full width at
half height, preferences evenly spaced on the circle, congruent tuning, reference heading 0,
amplitude 30 spikes/s on a baseline of 10 spikes/s over 1 s (the claims file's readings). The two
rsignal readings (8 headings, 1-degree grid) are run. As variants: other amplitudes and baselines,
Fisher information without the trace term of equation 3, and the coefficients fitted here
(fig5-regression-coefficients, reading c).

In [ ]:
print(f"Wrapped-Gaussian sigma for a 135 degree full width at half height: {stc1.SIGMA:.2f} degrees")
sizes = [16, 64, 256, 1000, 2000]
rsig_readings = {"8 headings": stc1.RSIGNAL_HEADINGS_8, "1-degree grid": stc1.RSIGNAL_HEADINGS_CONT}
rows = []
for name, H in rsig_readings.items():
    for n in sizes:
        prefs = stc1.uniform_prefs(n)
        ind = stc1.threshold(prefs, None, H)
        nv = stc1.threshold(prefs, stc1.NAIVE, H)
        tr = stc1.threshold(prefs, stc1.TRAINED, H)
        rows.append({"rsignal": name, "N": n, "independent": ind, "naive": nv, "trained": tr,
                     "naive/independent": nv / ind, "trained/naive": tr / nv})
fig6a = pd.DataFrame(rows)
print(fig6a.round(4).to_string(index=False))
print("\nExpected at N = 2000: naive/independent 5 +- 0.5 (4.5 to 5.5), trained within 10% of naive")
p_check = stc1.uniform_prefs(256)
d_rs = np.abs(stc1.signal_corr(p_check, stc1.RSIGNAL_HEADINGS_8) - stc1.signal_corr(p_check, stc1.RSIGNAL_HEADINGS_CONT)).max()
print(f"Largest difference in rsignal between the two readings (256 neurons): {d_rs:.1e}")


In [ ]:
coef_c = fits["c: slopes of the naive fit"]
variants = []
prefs = stc1.uniform_prefs(2000)
for label, kw, cn, ct in [
        ("claims reading (A 30, baseline 10)", {}, stc1.NAIVE, stc1.TRAINED),
        ("A 17, baseline 10", {"amplitude": 17.0}, stc1.NAIVE, stc1.TRAINED),
        ("A 35, baseline 10", {"amplitude": 35.0}, stc1.NAIVE, stc1.TRAINED),
        ("A 30, baseline 2", {"baseline": 2.0}, stc1.NAIVE, stc1.TRAINED),
        ("A 30, baseline 20", {"baseline": 20.0}, stc1.NAIVE, stc1.TRAINED),
        ("no trace term", {"trace": False}, stc1.NAIVE, stc1.TRAINED),
        ("coefficients fitted here (reading c)", {}, tuple(coef_c[[0, 1, 2]]), (coef_c[0], coef_c[1], coef_c[3]))]:
    ind = stc1.threshold(prefs, None, **kw)
    nv = stc1.threshold(prefs, cn, **kw)
    tr = stc1.threshold(prefs, ct, **kw)
    variants.append({"variant (N = 2000, 8 headings)": label, "naive/independent": nv / ind, "trained/naive": tr / nv})
print(pd.DataFrame(variants).round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(4, 3))
d = fig6a[fig6a.rsignal == "8 headings"]
for col, style in (("independent", "k--"), ("naive", "b-"), ("trained", "r--")):
    ax.loglog(d.N, d[col], style, label=col)
ax.set_xlabel("population size"); ax.set_ylabel("threshold (deg)"); ax.legend(); plt.show()

**Result: reproduced, resting on the claims file's tuning amplitude and baseline, which the paper
does not give.** At 2000 neurons the naive threshold is 5.41 times the independent one (4.5 to 5.5),
and the trained threshold is within 0.01% of the naive one, under both rsignal readings. The two readings give the same values because their rsignal
differs by at most 5.6e-07. The ratio
depends on the unstated parameters: 4.32 to 5.90 over the amplitude and baseline variants. It is
9.18 without the trace term of equation 3.

## fig6b-slope-not-intercept

> Population thresholds (256 neurons) are very sensitive to the slope of the relation between noise
> and signal correlation and only weakly sensitive to its intercept; a uniform increase in rnoise
> lowers the threshold slightly.

Grid of total slope (a_vestibular + a_visual, the slope on rsignal with congruent tuning) 0 to 0.4
and intercept 0 to 0.2, steps of 0.02 (the grid is the claims file's choice). Every point keeps
rnoise inside [-1, 1].

In [ ]:
slopes = np.round(np.arange(0.0, 0.4001, 0.02), 3)
intercepts = np.round(np.arange(0.0, 0.2001, 0.02), 3)
prefs256 = stc1.uniform_prefs(256)
fig6b = {}
for name, H in rsig_readings.items():
    grid = np.array([[stc1.threshold(prefs256, (s, 0.0, b), H) for s in slopes] for b in intercepts])
    fig6b[name] = grid
    slope_m = 0.12 + 0.091
    i_m = np.argmin(np.abs(slopes - 0.22))
    print(f"rsignal over {name}: threshold range over the grid {grid.min():.3f} to {grid.max():.3f} deg")
    for b_label, b in (("0.072 (naive)", 0.072), ("0.005 (trained)", 0.005)):
        row = np.array([stc1.threshold(prefs256, (s, 0.0, b), H) for s in slopes])
        print(f"  over slope 0 to 0.4 at intercept {b_label}: {row.min():.3f} to {row.max():.3f} (change {row.max() - row.min():.3f})")
        lo, hi = stc1.threshold(prefs256, (slope_m - 0.1, 0.0, b), H), stc1.threshold(prefs256, (slope_m + 0.1, 0.0, b), H)
        print(f"  over slope {slope_m - 0.1:.3f} to {slope_m + 0.1:.3f} (width 0.2) at intercept {b_label}: {lo:.3f} to {hi:.3f} (change {abs(hi - lo):.3f})")
    col = np.array([stc1.threshold(prefs256, (slope_m, 0.0, b), H) for b in intercepts])
    print(f"  over intercept 0 to 0.2 (width 0.2) at the measured slope {slope_m}: {col[0]:.3f} to {col[-1]:.3f} (change {col[-1] - col[0]:+.3f});"
          f" decreasing throughout: {bool(np.all(np.diff(col) < 0))}")
    print(f"  at slope 0.4: intercept 0 -> 0.2 changes threshold {grid[0, -1]:.3f} -> {grid[-1, -1]:.3f}")
    print("  threshold at the measured slope by intercept:", dict(zip(intercepts.tolist(), np.round(col, 4).tolist())))
    print(f"  at the trained and naive intercepts (0.005, 0.072): {stc1.threshold(prefs256, (slope_m, 0.0, 0.005), H):.4f}, {stc1.threshold(prefs256, (slope_m, 0.0, 0.072), H):.4f}")
fig, ax = plt.subplots(figsize=(4.5, 3.2))
cs = ax.contourf(slopes, intercepts, fig6b["8 headings"], 20)
plt.colorbar(cs, label="threshold (deg)")
ax.plot([0.211], [0.072], "bo"); ax.plot([0.211], [0.005], "wo")
ax.set_xlabel("slope"); ax.set_ylabel("intercept"); plt.show()

**Result: reproduced, resting on the same tuning parameters.** At 256 neurons, changing the slope
over a width of 0.2 around the measured 0.211 changes the threshold by 1.97 degrees (naive
intercept) and 1.93 (trained). Changing the intercept over 0 to 0.2 at the measured slope changes
it by 0.11 degrees. The threshold falls as the intercept rises from 0.02 to 0.2 (7.254 to 7.107), and
from the trained to the naive intercept (7.251 to 7.217). It rises slightly between 0 and 0.02
(7.220 to 7.254).

## fig7a-pair-sign-of-effect

> For a single pair of neurons, reducing noise correlation lowers the discrimination threshold when
> signal correlation is positive and raises it when signal correlation is negative.

One neuron prefers -90 degrees, the other's preference is swept from -90 to 90 in 1 degree steps
(-89.9 for the first, so the two curves differ), tuning as in fig6a.

In [ ]:
rows = []
for name, H in rsig_readings.items():
    for p2 in np.arange(-90.0, 90.01, 1.0):
        prefs = np.array([-90.0, p2 if p2 != -90 else -89.9])
        r = stc1.signal_corr(prefs, H)[0, 1]
        rows.append({"rsignal_grid": name, "pref2": p2, "rsignal": r,
                     "naive": stc1.threshold(prefs, stc1.NAIVE, H), "trained": stc1.threshold(prefs, stc1.TRAINED, H)})
fig7a = pd.DataFrame(rows)
fig7a["trained_lower"] = fig7a.trained < fig7a.naive
for name, d in fig7a.groupby("rsignal_grid"):
    pos, neg = d[d.rsignal > 0], d[d.rsignal < 0]
    print(f"{name}: {len(pos)} pairs with rsignal > 0, trained lower in {int(pos.trained_lower.sum())}; "
          f"{len(neg)} with rsignal < 0, trained higher in {int((~neg.trained_lower).sum())}")
    print(f"   trained/naive range, rsignal > 0: {(pos.trained / pos.naive).min():.4f} to {(pos.trained / pos.naive).max():.4f}; "
          f"rsignal < 0: {(neg.trained / neg.naive).min():.4f} to {(neg.trained / neg.naive).max():.4f}")
exceptions = fig7a[((fig7a.rsignal > 0) & ~fig7a.trained_lower) | ((fig7a.rsignal < 0) & fig7a.trained_lower)]
print("Pairs against the rule:")
print(exceptions.round(4).to_string(index=False))
print(fig7a[fig7a.rsignal_grid == "8 headings"].iloc[::15].round(4).to_string(index=False))
print("\nWhere trained/naive crosses 1 (linear interpolation between neighbouring pairs):")
for name, d in fig7a.groupby("rsignal_grid"):
    d = d.sort_values("rsignal")
    x, y = d.rsignal.values, (d.trained / d.naive).values - 1
    k = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]
    for j in k:
        cross = x[j] - y[j] * (x[j + 1] - x[j]) / (y[j + 1] - y[j])
        print(f"  {name}: rsignal = {cross:.4f} (between pref2 {d.pref2.values[j]:g} and {d.pref2.values[j + 1]:g} degrees)")


**Result: partially reproduced, resting on the same tuning parameters.** For all 88 pairs with
positive rsignal the trained threshold is lower, and for 92 of 93 with negative rsignal it is
higher. The exception has rsignal = -0.011 (second preference -2 degrees), where trained is 0.09%
lower than naive: the sign changes just below rsignal = 0, not at it.
Trained/naive crosses 1 at rsignal = -0.0216 (second preference between -1 and -2 degrees) under
both rsignal readings. The criterion asks for every
pair.

## fig7c-narrow-preferences

> In a 256-neuron population, narrowing heading preferences to give more pairs with positive signal
> correlation (49%, 75%, 94%) makes the weaker noise correlations of trained animals improve coding,
> but at best the trained threshold is about 8% lower than the naive.

256 neurons with preferences evenly spaced over ranges centred on 90 degrees, from the full circle
to 10 degrees wide, including [0, 180] and [30, 150] of the Figure 7C legend.

In [ ]:
widths = [360, 300, 240, 180, 150, 120, 90, 60, 30, 10]
rows = []
for name, H in rsig_readings.items():
    for w in widths:
        lo, hi = (-180.0, 180.0) if w == 360 else (90 - w / 2, 90 + w / 2)
        prefs = stc1.uniform_prefs(256, lo, hi)
        R = stc1.signal_corr(prefs, H)
        iu = np.triu_indices(256, 1)
        nv, tr = stc1.threshold(prefs, stc1.NAIVE, H), stc1.threshold(prefs, stc1.TRAINED, H)
        rows.append({"rsignal": name, "range": f"[{lo:g}, {hi:g}]", "positive_rsignal_%": 100 * (R[iu] > 0).mean(),
                     "independent": stc1.threshold(prefs, None, H), "naive": nv, "trained": tr,
                     "improvement_%": 100 * (1 - tr / nv)})
fig7c = pd.DataFrame(rows)
print(fig7c.round(3).to_string(index=False))
for name, d in fig7c.groupby("rsignal"):
    print(f"\n{name}: best improvement {d['improvement_%'].max():.2f}% (expected 8 +- 0.8) at {d.loc[d['improvement_%'].idxmax(), 'range']}; "
          f"improvement non-decreasing as the range narrows: {bool(np.all(np.diff(d['improvement_%'].values) >= -1e-9))}")
    print("  positive rsignal at [-180, 180], [0, 180], [30, 150]:",
          d.set_index("range").loc[["[-180, 180]", "[0, 180]", "[30, 150]"], "positive_rsignal_%"].round(1).tolist(), "(expected 49 +- 4.9, 75 +- 7.5, 94 +- 9.4)")
print("\nVariants of the best improvement (8 headings):")
for label, kw in [("A 17, baseline 10", {"amplitude": 17.0}), ("A 30, baseline 2", {"baseline": 2.0}),
                  ("A 30, baseline 20", {"baseline": 20.0}), ("no trace term", {"trace": False})]:
    imp = []
    for w in widths:
        lo, hi = (-180.0, 180.0) if w == 360 else (90 - w / 2, 90 + w / 2)
        prefs = stc1.uniform_prefs(256, lo, hi)
        imp.append(100 * (1 - stc1.threshold(prefs, stc1.TRAINED, **kw) / stc1.threshold(prefs, stc1.NAIVE, **kw)))
    print(f"  {label}: " + ", ".join(f"{w}: {v:.2f}%" for w, v in zip(widths, imp)))

**Result: partially reproduced, resting on the same tuning parameters.** The proportions of
positive rsignal are 48.6%, 73.1% and 92.4% for [-180, 180], [0, 180] and [30, 150], all within
tolerance. The trained improvement over naive grows from -0.48% to 4.83% as the range narrows, and
then levels off (4.58% to 4.83%). Its largest value, 4.83%, is below 8 +- 0.8%. Under the variants
the largest improvement is 3.6% to 6.0% with other amplitudes or baselines, and 11.75% without the
trace term. The effect is present and grows as the claim says, but its size at best is smaller.

## fig4dh-lateral-more-sensitive

> Neurons preferring lateral headings are more sensitive to heading variation around straight
> ahead than neurons preferring fore-aft headings (p << 0.001, factorial ANOVA).

The paper's single-electrode database (992 visual and 556 vestibular neurons, naive and trained)
is not in the data set. The experiment 1 tuning curves of 129 units of two trained monkeys are,
so the preference-group effect can be tested within trained animals only, without the training
factor of the paper's ANOVA. Per unit and condition, the threshold at straight ahead from
equations 2 and 3 for one neuron with variance equal to the mean, on the tuning curve
interpolated by a periodic cubic spline or linearly (the claims file's readings); preferred
heading by the vector sum or the spline peak (the claims file's readings; the vector sum weights
each response by the angle it covers, as the headings are unevenly spaced, see `stc1.py`).
ANOVA on threshold, raw and log10, per condition (preference group alone) and with condition as
a second factor (preference group, condition and their interaction, Type II). A unit whose
curve is flat at 0 has an infinite threshold and is left out; the count is printed. As a
variant, not a reading of the claims file, the neuronal thresholds that experiment 2 holds for
the same cells (neurometric thresholds of Gu et al. 2008, from responses to headings near
straight ahead), and the vector sum over the 8 evenly spaced headings alone. First, a check of the two preference readings on synthetic tuning curves
sampled at the experiment 1 headings.

In [ ]:
H1 = mat["experiment1"].units[0].ves.stim_global
print("Synthetic wrapped-Gaussian curves (fig6a tuning) at the experiment 1 headings:")
print("  true preference -> weighted vector sum, spline peak, unweighted vector sum")
for p_true in (0, 30, 60, 90, 120, 150, -45, -100):
    r = stc1.tuning([p_true], H1)[0]
    h = np.deg2rad(H1)
    unweighted = np.rad2deg(np.arctan2((r * np.sin(h)).sum(), (r * np.cos(h)).sum()))
    print(f"  {p_true:5d} -> {stc1.pref_vector_sum(H1, r):7.1f} {stc1.pref_spline_peak(H1, r):7.1f} {unweighted:7.1f}")


In [ ]:
rows = []
even8 = np.isin(H1, np.arange(-180, 180, 45))
for u1, u2 in zip(mat["experiment1"].units, mat["experiment2"].units):
    for cond in ("ves", "vis"):
        c1 = getattr(u1, cond)
        rows.append({"cell": u1.file_id, "monkey": stc1.monkey_of(u1.file_id), "condition": cond,
                     "pref_vector_sum": stc1.pref_vector_sum(c1.stim_global, c1.resp_global),
                     "pref_spline_peak": stc1.pref_spline_peak(c1.stim_global, c1.resp_global),
                     "variant_pref_vector_sum_8": stc1.pref_vector_sum(c1.stim_global[even8], c1.resp_global[even8]),
                     "thr_spline": stc1.neuron_threshold(c1.stim_global, c1.resp_global, "spline"),
                     "thr_linear": stc1.neuron_threshold(c1.stim_global, c1.resp_global, "linear"),
                     "variant_thr_experiment2": float(getattr(u2, cond).thresh)})
sens = pd.DataFrame(rows).replace(np.inf, np.nan)
thr_cols = ["thr_spline", "thr_linear", "variant_thr_experiment2"]
print("Values:", len(sens), "; infinite thresholds left out:", sens[thr_cols].isna().sum().to_dict())
print("Threshold ranges (degrees):")
print(sens[thr_cols].describe().loc[["count", "min", "50%", "max"]].round(2).to_string())
print("Spearman rho of the Fisher thresholds with the experiment 2 thresholds:",
      {c: round(stats.spearmanr(sens[c], sens.variant_thr_experiment2, nan_policy="omit")[0], 3) for c in thr_cols[:2]})
res = []
for pref in ("pref_vector_sum", "pref_spline_peak", "variant_pref_vector_sum_8"):
    sens["pgroup"] = stc1.preference_group(sens[pref])
    for thr in thr_cols:
        for scale in ("raw", "log10"):
            d = sens.dropna(subset=[thr]).copy()
            d["y"] = d[thr] if scale == "raw" else np.log10(d[thr])
            row = {"preference": pref, "threshold": thr, "scale": scale}
            for cond in ("ves", "vis"):
                dc = d[d.condition == cond]
                a = sm.stats.anova_lm(smf.ols("y ~ C(pgroup)", dc).fit(), typ=2)
                row[f"{cond}_n_fa/lat"] = f"{(dc.pgroup == 'fore-aft').sum()}/{(dc.pgroup == 'lateral').sum()}"
                row[f"{cond}_median_fa"] = dc[thr][dc.pgroup == "fore-aft"].median()
                row[f"{cond}_median_lat"] = dc[thr][dc.pgroup == "lateral"].median()
                row[f"{cond}_p"] = a.loc["C(pgroup)", "PR(>F)"]
            a2 = sm.stats.anova_lm(smf.ols("y ~ C(pgroup) * C(condition)", d).fit(), typ=2)
            row["two_factor_p_group"] = a2.loc["C(pgroup)", "PR(>F)"]
            row["two_factor_p_interaction"] = a2.loc["C(pgroup):C(condition)", "PR(>F)"]
            row["lateral_lower_both"] = bool((row["ves_median_lat"] < row["ves_median_fa"]) and (row["vis_median_lat"] < row["vis_median_fa"]))
            res.append(row)
fig4dh = pd.DataFrame(res)
print(fig4dh.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
crit = fig4dh[(fig4dh.threshold != "variant_thr_experiment2") & ~fig4dh.preference.str.startswith("variant")]
passes = crit[((crit.two_factor_p_group < 0.001) | ((crit.ves_p < 0.001) & (crit.vis_p < 0.001))) & crit.lateral_lower_both]
print("\nReadings of the claims file with lateral lower at p < 0.001:", len(passes), "of", len(crit))
print(passes[["preference", "threshold", "scale", "ves_p", "vis_p", "two_factor_p_group"]].to_string(index=False))
print("\nPer monkey (vector-sum preference, spline threshold, log10): median threshold fore-aft, lateral, p")
sens["pgroup"] = stc1.preference_group(sens.pref_vector_sum)
for (m, c), d in sens.groupby(["monkey", "condition"]):
    fa, la = np.log10(d.thr_spline[d.pgroup == "fore-aft"]), np.log10(d.thr_spline[d.pgroup == "lateral"])
    print(f"  monkey {m}, {c}: {10 ** fa.median():.1f} (n {len(fa)}), {10 ** la.median():.1f} (n {len(la)}), p = {stats.ttest_ind(fa, la).pvalue:.3g}")


**Result: reproduced, on a subset of the paper's data (258 values from 129 units of two trained
monkeys, against 992 and 556 neurons of naive and trained animals) and without the training factor
of the paper's ANOVA.** On synthetic curves the weighted vector sum and the spline peak recover the
true preference to within 3 degrees, and the unweighted vector sum does not (30 degrees comes out
as 16). Lateral units have lower median thresholds than fore-aft ones under every reading and
variant, in both conditions. Under one of the 8 readings of the claims file the difference reaches
p < 0.001: vector-sum preference, spline threshold, log10, with condition as a factor (p = 1.9e-07).
In that reading, per condition, visual gives p = 3.4e-07 and vestibular p = 0.040, so the
per-condition reading fails in the vestibular condition. In the other 7 readings the two-factor
p ranges from 0.013 to 0.77, and no per-condition reading has p < 0.001 in both conditions. With the spline-peak preference no reading reaches p < 0.05. The neurometric thresholds of
experiment 2, a variant, give two-factor p = 7e-07 (log10, vector sum) and p = 0.0011 (log10, spline
peak). Per monkey, the visual difference is significant in both (p = 0.00024 and 0.00017) and the
vestibular one in neither (p = 0.082 and 0.185). The linear reading leaves out 2 values whose curve
is flat at 0.

## fig8ab-choice-probability-lateral

> Neurons preferring fore-aft headings have smaller choice probabilities than neurons preferring
> lateral headings (p = 0.019, t test), and choice probability does not differ between neurons
> preferring leftward and rightward headings (p = 0.11).

The paper's database has 311 values from single-electrode recordings, vestibular and visual pooled.
The data set holds the choice probabilities of 129 units from two trained monkeys (experiment 2,
Gu et al. 2008, one of the studies the paper draws on), in both conditions: 258 values, a subset
whose overlap with the paper's 311 cannot be checked. Each unit's preferred heading in a condition
comes from its experiment 1 tuning curve of the same condition (the same cells, recorded back to
back), by the two readings of the claims file, as in fig4dh-lateral-more-sensitive (the vector sum
weights each response by the angle it covers; the round-1 version did not, which pulled
preferences toward straight ahead). Leftward is a preference below 0 degrees (the
paper puts leftward at -90). Whether the leftward-rightward test covers all units or only the
lateral ones is not stated; both are run. Two-sample Student t tests, with Welch as a variant. As
further variants, not readings of the claims file: the vector sum over the 8 evenly spaced
headings only, and the distance of CP from 0.5 (the Figure 8A legend speaks of CPs "more deviated away from the
chance level of 0.5"; the data set does not say in which direction its CPs are signed; the share of CPs above 0.5 is
printed as a check of the sign).

In [ ]:
rows = []
even = np.isin(mat["experiment1"].units[0].ves.stim_global, np.arange(-180, 180, 45))  # variant: drop +-22.5
for u1, u2 in zip(mat["experiment1"].units, mat["experiment2"].units):
    for cond in ("ves", "vis"):
        c1, c2 = getattr(u1, cond), getattr(u2, cond)
        rows.append({"cell": u1.file_id, "monkey": stc1.monkey_of(u1.file_id), "condition": cond, "cp": float(c2.cp),
                     "pref_vector_sum": stc1.pref_vector_sum(c1.stim_global, c1.resp_global),
                     "pref_spline_peak": stc1.pref_spline_peak(c1.stim_global, c1.resp_global),
                     "pref_vector_sum_8": stc1.pref_vector_sum(c1.stim_global[even], c1.resp_global[even])})
cps = pd.DataFrame(rows)
print("Values:", len(cps), "from", cps.cell.nunique(), "units; per monkey and condition:", cps.groupby(["monkey", "condition"]).size().to_dict())
print("CP range", cps.cp.min(), "to", cps.cp.max(), "; missing", int(cps.cp.isna().sum()),
      "; share above 0.5", round(float((cps.cp > 0.5).mean()), 3), "; mean", round(cps.cp.mean(), 4))
res = []
cps["cp_deviation"] = (cps.cp - 0.5).abs()
print("Agreement of the readings' groups: vector sum and spline peak",
      round(float((stc1.preference_group(cps.pref_vector_sum) == stc1.preference_group(cps.pref_spline_peak)).mean()), 3),
      "; weighted vector sum over all 10 and vector sum over the 8 evenly spaced headings",
      round(float((stc1.preference_group(cps.pref_vector_sum) == stc1.preference_group(cps.pref_vector_sum_8)).mean()), 3))
for reading in ("pref_vector_sum", "pref_spline_peak", "pref_vector_sum_8"):
    grp = stc1.preference_group(cps[reading])
    fa, la = cps.cp[grp == "fore-aft"], cps.cp[grp == "lateral"]
    left_all, right_all = cps.cp[cps[reading] < 0], cps.cp[cps[reading] >= 0]
    lat = cps[grp == "lateral"]
    left_lat, right_lat = lat.cp[lat[reading] < 0], lat.cp[lat[reading] >= 0]
    res.append({"preference": reading, "n_fore_aft": len(fa), "cp_fore_aft": fa.mean(), "n_lateral": len(la), "cp_lateral": la.mean(),
                "p_fa_vs_lat": stats.ttest_ind(fa, la).pvalue, "p_fa_vs_lat_welch": stats.ttest_ind(fa, la, equal_var=False).pvalue,
                "cp_left_all": left_all.mean(), "cp_right_all": right_all.mean(), "n_left_all": len(left_all), "n_right_all": len(right_all),
                "p_left_right_all": stats.ttest_ind(left_all, right_all).pvalue,
                "cp_left_lateral": left_lat.mean(), "cp_right_lateral": right_lat.mean(),
                "p_left_right_lateral": stats.ttest_ind(left_lat, right_lat).pvalue,
                "variant_dev_fore_aft": cps.cp_deviation[grp == "fore-aft"].mean(), "variant_dev_lateral": cps.cp_deviation[grp == "lateral"].mean(),
                "variant_p_dev": stats.ttest_ind(cps.cp_deviation[grp == "fore-aft"], cps.cp_deviation[grp == "lateral"]).pvalue})
fig8ab = pd.DataFrame(res)
print(fig8ab.round(4).T.to_string())
print("\nPer monkey and condition (vector-sum preference): mean CP fore-aft, lateral, t-test p")
for (m, c), d in cps.groupby(["monkey", "condition"]):
    g = stc1.preference_group(d.pref_vector_sum)
    print(f"  monkey {m}, {c}: {d.cp[g == 'fore-aft'].mean():.3f} (n {int((g == 'fore-aft').sum())}), "
          f"{d.cp[g == 'lateral'].mean():.3f} (n {int((g == 'lateral').sum())}), p = {stats.ttest_ind(d.cp[g == 'fore-aft'], d.cp[g == 'lateral']).pvalue:.3f}")

**Result: not reproduced, on a subset of the paper's data (258 values from 129 units of two trained
monkeys, against 311).** Fore-aft units do not have smaller choice probabilities than lateral ones.
With preferences from the spline peak, fore-aft 0.527 against lateral 0.542, p = 0.136. With the
weighted vector sum, fore-aft 0.533 against lateral 0.535, p = 0.796. Leftward and rightward units
do not differ under any reading (p = 0.25 to 0.35), as the paper reports. In the variants, the
vector sum over the 8 evenly spaced headings gives p = 0.691, and the distance of CP from 0.5 gives
p = 0.10 to 0.17. The vector sum and the spline peak put 71.7% of values in the same group. Of the
CPs, 71.7% are above 0.5 (mean 0.534), as expected if they are signed relative to each unit's
preferred direction; the data set does not say so.

## fig8cd-noise-not-preference

> Noise correlation does not depend significantly on heading preference (p = 0.2, t test); lateral
> neurons have slightly smaller noise correlations than fore-aft neurons.

Each pair entered twice, at each unit's preferred heading (the data set's `heading_pref`), with
the pooled rnoise; t test of lateral against fore-aft. Of the claims file's readings, only the
trained pairs under 1 mm (178 points) can be formed from the data set: the other trained pairs, the
pairs over 1 mm and the 55 discrimination-task points are not in it. It is therefore the reading
closest to the legend's 328 and 55 that can be run, and the outcome is judged on it; all 127
pairs (254 points) are shown as a variant. The 127 pairs are not one of the claims file's
readings: they are its "all 179 pairs" reading with the 52 pairs over 1 mm missing. Their 254
points are nearer 328 than the 178 points used, so a reading of the claims file's rule by point
count alone would judge on them. The outcome is judged on the trained pairs because that sample
is a complete reading of the criterion and the Results speak of trained animals (Possible
rsignal Distributions in Area MSTd, paragraph 3); the variant's result is reported with the
conclusion.

In [ ]:
ut = stc1.unit_table(pairs)
rows = []
for sample, d in (("trained pairs under 1 mm", ut[ut.group == "trained"]), ("all 127 pairs (variant)", ut)):
    for cond in ("vis", "ves"):
        g = stc1.preference_group(d[f"pref_{cond}"])
        fa, la = d.rnoise[g == "fore-aft"], d.rnoise[g == "lateral"]
        rows.append({"sample": sample, "preference": cond, "points": len(d), "n_fore_aft": len(fa), "rnoise_fore_aft": fa.mean(),
                     "sem_fa": fa.sem(), "n_lateral": len(la), "rnoise_lateral": la.mean(), "sem_lat": la.sem(),
                     "p_student": stats.ttest_ind(fa, la).pvalue, "p_welch": stats.ttest_ind(fa, la, equal_var=False).pvalue})
fig8cd = pd.DataFrame(rows)
print(fig8cd.round(4).to_string(index=False))
print("\nPaper: p = 0.2, lateral slightly below fore-aft; Figure 8C legend: 328 fixation and 55 task points")

**Result: reproduced, resting on the pooled-rnoise assumption.** Over the 178 points of the
trained pairs under 1 mm, the reading nearest the legend's counts that the data set allows,
noise correlation does not differ between lateral and fore-aft units. With visual preferences
lateral 0.033 against fore-aft 0.014, p = 0.434. With vestibular preferences lateral 0.011 against
0.045, p = 0.170. Lateral is lower only with vestibular preferences. With all 127 pairs (254 points)
and vestibular preferences, the difference is significant (0.022 against 0.096, p = 0.0009); with
visual preferences it is not (p = 0.623). That variant's 254 points are nearer the legend's 328
than the 178 used, so judged by point count alone, the claims file's rule would point to it. It is
not one of the rule's readings (it is the all-179 reading with 52 pairs missing), and the outcome
is kept on the trained pairs, which the Results name. Under the variant the claim would not hold
with vestibular preferences.

## Summary

| Claim | Outcome |
|---|---|
| training-group-link | reproduced |
| fig2b-trained-lower-noise-correlation | reproduced (assumed) |
| figs2b-individual-animals | reproduced (assumed; means only) |
| fig1-example-pair | reproduced |
| fig1f-modality-correlated | partially reproduced (127 of 179 pairs) |
| fig1f-modality-means-equal | reproduced (127 of 179 pairs) |
| fig5-noise-signal-positive | reproduced (assumed) |
| fig5-slope-unchanged | reproduced (assumed) |
| fig5-intercept-lower-trained | partially reproduced (assumed) |
| fig5-regression-coefficients | partially reproduced (assumed) |
| fig6a-population-thresholds | reproduced (assumed) |
| fig6b-slope-not-intercept | reproduced (assumed) |
| fig7a-pair-sign-of-effect | partially reproduced (assumed) |
| fig7c-narrow-preferences | partially reproduced (assumed) |
| fig4dh-lateral-more-sensitive | reproduced (subset, trained animals only) |
| fig8ab-choice-probability-lateral | not reproduced (subset) |
| fig8cd-noise-not-preference | reproduced (assumed) |
| fig2c, figs2a, fig3 (two claims), figs8c, discussion-stimulus-reduces-correlation | blocked, data missing (no spike times or trial responses) |
| figs1-distance | blocked, data missing (no distances, no pairs over 1 mm) |
| methods-single-vs-multiple-electrodes | blocked, metadata insufficient (no electrode label) |
| fig2a-behavioral-learning | blocked, data missing (no training histories) |
| fig4 (three claims), fig4dh-training-no-sensitivity-change, figs4 | blocked, data missing (no naive single units) |
| fig7b-msdt-pairs, figs5-congruency | blocked, data missing (no tuning curves of the pair units) |
| figs6-task-independent | blocked, data missing |
| figs8ab-eye-movements | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The data set holds per-condition noise correlations,
not the pooled one, so the pooled value is taken as the mean of the two. Under that reading the
main result matches the paper to the third decimal. Training groups come from the paper's
per-animal pair counts. Only the 127 pairs under 1 mm are available, where Figure 1F uses 179.
The simulations use a tuning amplitude and baseline that the paper does not give, and the
threshold ratios depend on them. Single-unit thresholds and choice probabilities come from 129 units
of two trained monkeys, not the paper's database of naive and trained animals (992 and 556
neurons, 311 choice probabilities), so the preference-group test of Figure 4 runs without the
training factor.

What was found about the data set. It holds derived values only: no spike times, trial
responses, distances, electrode labels or training-group labels, and none of the 52 pairs over
1 mm. The data descriptor lists seven monkeys for the MSTd pairs. The files name eight,
including monkey 24 with two pairs. One recording (m19c21r1) has six pairs from four units,
where the paper states a maximum of five pairs per experiment. How the preferred headings
were computed is not described. The pairs are internally consistent (units per recording,
rsignal against the preference difference) and match the paper's group counts of pairs and
units.